# EXP-2026-001: Catastrophic Forgetting & Stability-Plasticity Dilemma

**Institute for Cognitive Intelligence Research (ICIR)**  
*Document / Experiment ID:* `EXP-2026-001` (Ledger ID: `EXP-2026-11-01-001`)  
*Investigator:* Chilaka Chigozie Micheal (Research Lead & Founder, ICIR)  
*Research Track:* Track A: Continual Learning & Dynamic Plasticity (ICIR 2026–2031 Horizon, Phase 1)  
*Formal Protocol:* [`experiments/protocols/EXP-2026-001-protocol.md`](../protocols/EXP-2026-001-protocol.md)  
*Status:* `PLANNED` / `STAGED_FOR_REVIEW`

---

## 1. Research Question & Falsifiable Hypotheses

### Research Question
> *To what quantitative degree does unconstrained sequential stochastic gradient descent induce catastrophic interference across sequentially presented orthogonal task distributions, and how wide is the empirical stability gap separating naive sequential adaptation from joint offline empirical risk minimization?*

### Falsifiable Hypotheses
- **Alternative Hypothesis ($H_1$):** In a 5-task sequential Permuted MNIST benchmark evaluated across the five canonical ICIR pseudo-random seeds ($\mathcal{S} = \{42, 1337, 2026, 7, 99\}$), a multi-layer perceptron trained via unregularized sequential gradient descent without rehearsal will experience catastrophic forgetting on Task 1 exceeding $70\%$ degradation ($BWT_{\text{naive}} \le -0.70$), while Joint Offline Training preserves average accuracy $A_{5, \text{joint}} \ge 0.90$ with $BWT_{\text{joint}} \ge -0.05$.
- **Null Hypothesis ($H_0$):** Sequential gradient descent without rehearsal retains prior task representations ($BWT_{\text{naive}} > -0.15$), rejecting the existence of catastrophic forgetting in this sequential parameter subspace.

---

## 2. Mathematical Foundations

Let $T$ tasks be presented sequentially with loss $\mathcal{L}_t(\theta)$ for task $t$. The gradient update direction is $\Delta \theta_t \propto -\nabla_\theta \mathcal{L}_t(\theta)$. When gradient directions between previous tasks $k < t$ and the current task $t$ are misaligned:
$$\langle \nabla_\theta \mathcal{L}_k(\theta), \nabla_\theta \mathcal{L}_t(\theta) \rangle < 0$$
optimization on task $t$ strictly increases the empirical risk on prior tasks, resulting in catastrophic forgetting.

### Evaluation Metrics
Given performance matrix $R \in \mathbb{R}^{T \times T}$ where $R_{i, j}$ is accuracy on task $j$ after completing task $i$:
1. **Average Accuracy ($A_T$):** $A_T = \frac{1}{T} \sum_{j=1}^T R_{T, j}$
2. **Backward Transfer ($BWT$):** $BWT = \frac{1}{T-1} \sum_{j=1}^{T-1} (R_{T, j} - R_{j, j})$
3. **Forward Transfer ($FWT$):** $FWT = \frac{1}{T-1} \sum_{j=2}^T R_{j-1, j}$

In [ ]:
import os
import sys
import time
import json
import random
from typing import Dict, List, Tuple
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset
import torchvision
import torchvision.transforms as transforms

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"PyTorch version    : {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

In [ ]:
# ==============================================================================
# Defensive GPU Detection & Environment Verification
# Verifies connection to Google Colab GPU runtime (Tesla T4) or falls back to CPU.
# ==============================================================================
cuda_available = torch.cuda.is_available()

print("=" * 65)
print("ICIR RUNTIME HARDWARE & ACCELERATOR AUDIT")
print("=" * 65)
print(f"CUDA Available     : {cuda_available}")

if cuda_available:
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
    device_prop = torch.cuda.get_device_properties(0)
    total_memory_gb = round(device_prop.total_memory / (1024 ** 3), 2)
    cuda_arch = torch.cuda.get_device_capability(0)
    
    print(f"Device Name        : {device_name}")
    print(f"Dedicated VRAM     : {total_memory_gb} GB")
    print(f"Compute Capability : SM {cuda_arch[0]}.{cuda_arch[1]}")
    print(f"CUDA Toolchain     : {torch.version.cuda}")
    print(f"cuDNN Version      : {torch.backends.cudnn.version()}")
    print("STATUS: Verified connected to Google Colab GPU Runtime (Tesla T4).")
else:
    device = torch.device("cpu")
    total_memory_gb = 0.0
    print("Device Name        : CPU (Local Fallback)")
    print("WARNING: CUDA is not available. Execution will proceed on CPU.")
print("=" * 65)

## 3. Reproducibility & Deterministic Seed Control

Per ICIR Empirical Rigor Pillar I (`experiments/README.md`), all continual learning benchmarks must be evaluated across the five canonical pseudo-random seeds:
$$\mathcal{S} = [42, 1337, 2026, 7, 99]$$

The function `seed_everything` configures deterministic guarantees across all runtime libraries.

In [ ]:
CANONICAL_SEEDS = [42, 1337, 2026, 7, 99]

def seed_everything(seed: int = 42) -> None:
    """Enforces strict determinism across Python, NumPy, PyTorch, and cuDNN."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Test initialization with primary seed
seed_everything(CANONICAL_SEEDS[0])
print(f"Deterministic seed controller initialized with seed {CANONICAL_SEEDS[0]}.")

In [ ]:
@dataclass
class ExperimentConfig:
    experiment_id: str = "EXP-2026-001"
    num_tasks: int = 5
    input_dim: int = 784             # 28x28 flattened MNIST pixels
    hidden_dims: Tuple[int, ...] = (256, 128)
    num_classes: int = 10
    dropout: float = 0.1
    batch_size: int = 128
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    epochs_per_task: int = 3
    seeds: Tuple[int, ...] = tuple(CANONICAL_SEEDS)
    # Target directory for metrics, logs, and figures
    output_dir: str = "experiments/outputs/EXP-2026-001"

config = ExperimentConfig()

print("=" * 65)
print("EXPERIMENT CONFIGURATION (EXP-2026-001)")
print("=" * 65)
for k, v in asdict(config).items():
    print(f"  {k:<20}: {v}")
print("=" * 65)

## 4. Benchmark Task Construction: Permuted MNIST (Domain-IL)

In Permuted MNIST, each task $t \in \{1, \dots, T\}$ applies an orthogonal fixed permutation $\pi_t \in \mathcal{S}_{784}$ to the input pixel space.
- Task 1: Identity permutation $\pi_1 = \text{Id}$ (canonical MNIST digits).
- Tasks $2 \dots T$: Independent pseudo-random permutations sampled deterministically from `seed`.

In [ ]:
def get_permuted_mnist_tasks(num_tasks: int, seed: int, data_root: str = "./data") -> List[Dict]:
    """
    Constructs a sequence of T permuted MNIST tasks.
    Returns list of dicts with keys: 'task_id', 'train', 'test', 'permutation'.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    os.makedirs(data_root, exist_ok=True)
    raw_train = torchvision.datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
    raw_test = torchvision.datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
    
    # Pre-normalize and flatten into contiguous tensors
    train_x = raw_train.data.view(-1, 784).float() / 255.0
    train_x = (train_x - 0.1307) / 0.3081
    train_y = raw_train.targets
    
    test_x = raw_test.data.view(-1, 784).float() / 255.0
    test_x = (test_x - 0.1307) / 0.3081
    test_y = raw_test.targets
    
    # Generate deterministic permutations
    task_rng = np.random.RandomState(seed)
    permutations = [np.arange(784)]  # Task 1 is identity
    for _ in range(1, num_tasks):
        permutations.append(task_rng.permutation(784))
        
    tasks = []
    for t_idx, perm in enumerate(permutations):
        perm_idx = torch.from_numpy(perm).long()
        t_train_x = train_x[:, perm_idx]
        t_test_x = test_x[:, perm_idx]
        
        train_ds = TensorDataset(t_train_x, train_y)
        test_ds = TensorDataset(t_test_x, test_y)
        tasks.append({
            "task_id": t_idx + 1,
            "train": train_ds,
            "test": test_ds,
            "permutation": perm
        })
        
    return tasks

print("Permuted MNIST task generator compiled successfully.")

In [ ]:
class ContinualMLP(nn.Module):
    """
    Multi-Layer Perceptron benchmark for continual learning.
    Architecture: Input(784) -> Linear(256) -> ReLU -> Dropout(0.1) 
                            -> Linear(128) -> ReLU -> Dropout(0.1) 
                            -> Linear(10)
    """
    def __init__(self, input_dim: int = 784, hidden_dims: Tuple[int, ...] = (256, 128),
                 num_classes: int = 10, dropout: float = 0.1):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

# Model initialization and parameter audit
test_mlp = ContinualMLP()
total_params = sum(p.numel() for p in test_mlp.parameters() if p.requires_grad)
print(f"Model architecture verified. Trainable parameter count: {total_params:,}")

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer,
                criterion: nn.Module, device: torch.device) -> float:
    """Performs a single training epoch and returns mean cross-entropy loss."""
    model.train()
    running_loss = 0.0
    total_samples = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        batch_size = x.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
    return running_loss / max(total_samples, 1)

@torch.no_grad()
def evaluate_accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    """Computes exact classification accuracy over the dataset loader."""
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return correct / max(total, 1)

def compute_continual_metrics(R: np.ndarray) -> Dict[str, float]:
    """Computes A_T, BWT, and FWT from accuracy matrix R in R^{T x T}."""
    T = R.shape[0]
    A_T = float(np.mean(R[T - 1, :]))
    
    if T > 1:
        bwt_vals = [R[T - 1, j] - R[j, j] for j in range(T - 1)]
        BWT = float(np.mean(bwt_vals))
        fwt_vals = [R[j - 1, j] for j in range(1, T)]
        FWT = float(np.mean(fwt_vals))
    else:
        BWT = 0.0
        FWT = 0.0
        
    return {
        "A_T": A_T,
        "BWT": BWT,
        "FWT": FWT,
        "R_matrix": R.tolist()
    }

print("Training, evaluation, and metrics engines compiled.")

In [ ]:
def run_naive_sequential(seed: int, cfg: ExperimentConfig, device: torch.device) -> Dict:
    """Executes Regime A: Naive Sequential Fine-Tuning (Empirical Lower Bound)."""
    seed_everything(seed)
    tasks = get_permuted_mnist_tasks(cfg.num_tasks, seed)
    model = ContinualMLP(cfg.input_dim, cfg.hidden_dims, cfg.num_classes, cfg.dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    R = np.zeros((cfg.num_tasks, cfg.num_tasks))
    start_time = time.time()
    
    for t_train in range(cfg.num_tasks):
        train_loader = DataLoader(tasks[t_train]["train"], batch_size=cfg.batch_size, shuffle=True)
        for epoch in range(cfg.epochs_per_task):
            train_epoch(model, train_loader, optimizer, criterion, device)
            
        # Evaluate performance on all tasks
        for t_eval in range(cfg.num_tasks):
            test_loader = DataLoader(tasks[t_eval]["test"], batch_size=cfg.batch_size * 2, shuffle=False)
            R[t_train, t_eval] = evaluate_accuracy(model, test_loader, device)
            
    wall_clock = time.time() - start_time
    metrics = compute_continual_metrics(R)
    metrics["wall_clock_seconds"] = wall_clock
    metrics["seed"] = seed
    metrics["regime"] = "Naive Sequential Fine-Tuning"
    return metrics

def run_joint_offline_oracle(seed: int, cfg: ExperimentConfig, device: torch.device) -> Dict:
    """Executes Regime B: Joint Offline Training Oracle (Empirical Upper Bound)."""
    seed_everything(seed)
    tasks = get_permuted_mnist_tasks(cfg.num_tasks, seed)
    model = ContinualMLP(cfg.input_dim, cfg.hidden_dims, cfg.num_classes, cfg.dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    joint_train_ds = ConcatDataset([t["train"] for t in tasks])
    joint_loader = DataLoader(joint_train_ds, batch_size=cfg.batch_size, shuffle=True)
    
    start_time = time.time()
    for epoch in range(cfg.epochs_per_task):
        train_epoch(model, joint_loader, optimizer, criterion, device)
        
    R = np.zeros((cfg.num_tasks, cfg.num_tasks))
    for t_eval in range(cfg.num_tasks):
        test_loader = DataLoader(tasks[t_eval]["test"], batch_size=cfg.batch_size * 2, shuffle=False)
        acc = evaluate_accuracy(model, test_loader, device)
        for row in range(cfg.num_tasks):
            R[row, t_eval] = acc
            
    wall_clock = time.time() - start_time
    metrics = compute_continual_metrics(R)
    metrics["wall_clock_seconds"] = wall_clock
    metrics["seed"] = seed
    metrics["regime"] = "Joint Offline Oracle"
    return metrics

print("Regime execution runners compiled successfully.")

## 5. Experiment Execution

Review the parameters and select execution mode below:
- **Option A (Full Benchmark):** Evaluates all 5 canonical seeds ($42, 1337, 2026, 7, 99$). On the connected NVIDIA Tesla T4 GPU, the full run finishes in approximately 90–120 seconds.
- **Option B (Fast Verification):** Evaluates single seed $42$ (finishes in ~15–20 seconds).

> [!NOTE]
> Execute the cell below on the connected Colab GPU kernel to begin training.

In [ ]:
# ==============================================================================
# MAIN EXPERIMENT TRIGGER
# ==============================================================================
RUN_ALL_CANONICAL_SEEDS = True  # Set to False for single seed (42) verification
active_seeds = config.seeds if RUN_ALL_CANONICAL_SEEDS else (config.seeds[0],)

print(f"Beginning EXP-2026-001 computational execution on: {device}")
print(f"Seeds to evaluate ({len(active_seeds)}): {list(active_seeds)}")
print("-" * 65)

results_naive = []
results_joint = []

for idx, s in enumerate(active_seeds, 1):
    print(f"[{idx}/{len(active_seeds)}] Executing Seed {s}...")
    
    # Regime A: Naive Sequential
    print("  -> Running Regime A: Naive Sequential Fine-Tuning...")
    res_a = run_naive_sequential(s, config, device)
    results_naive.append(res_a)
    print(f"     Terminal A_T: {res_a['A_T']:.4f} | BWT: {res_a['BWT']:.4f} | Time: {res_a['wall_clock_seconds']:.2f}s")
    
    # Regime B: Joint Offline Oracle
    print("  -> Running Regime B: Joint Offline Oracle...")
    res_b = run_joint_offline_oracle(s, config, device)
    results_joint.append(res_b)
    print(f"     Terminal A_T: {res_b['A_T']:.4f} | BWT: {res_b['BWT']:.4f} | Time: {res_b['wall_clock_seconds']:.2f}s")

print("-" * 65)
print("Execution phase complete.")

In [ ]:
def aggregate_regime_statistics(results: List[Dict]) -> Dict:
    a_t = [r["A_T"] for r in results]
    bwt = [r["BWT"] for r in results]
    fwt = [r["FWT"] for r in results]
    wtime = [r["wall_clock_seconds"] for r in results]
    
    return {
        "A_T_mean": float(np.mean(a_t)),
        "A_T_std": float(np.std(a_t)),
        "BWT_mean": float(np.mean(bwt)),
        "BWT_std": float(np.std(bwt)),
        "FWT_mean": float(np.mean(fwt)),
        "FWT_std": float(np.std(fwt)),
        "time_mean": float(np.mean(wtime)),
        "time_std": float(np.std(wtime)),
        "individual_runs": results
    }

summary_naive = aggregate_regime_statistics(results_naive)
summary_joint = aggregate_regime_statistics(results_joint)

print("=" * 75)
print(f"EXP-2026-001 EMPIRICAL RESULTS SUMMARY (N = {len(active_seeds)} SEEDS)")
print("=" * 75)
print(f"{'Experimental Regime':<26} | {'Terminal Acc (A_T)':<18} | {'Backward Transfer':<18} | {'Avg Time (s)':<10}")
print("-" * 75)
print(f"{'Naive Sequential (Lower Bound)':<26} | {summary_naive['A_T_mean']:.4f} ± {summary_naive['A_T_std']:.4f}   | {summary_naive['BWT_mean']:.4f} ± {summary_naive['BWT_std']:.4f}   | {summary_naive['time_mean']:<6.2f}s")
print(f"{'Joint Oracle (Upper Bound)':<26} | {summary_joint['A_T_mean']:.4f} ± {summary_joint['A_T_std']:.4f}   | {summary_joint['BWT_mean']:.4f} ± {summary_joint['BWT_std']:.4f}   | {summary_joint['time_mean']:<6.2f}s")
print("=" * 75)

In [ ]:
# Visualizations: Retention Matrix Heatmap & Task Degradation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Retention Matrix Heatmap (Seed 42)
sample_R = np.array(results_naive[0]["R_matrix"])
im = axes[0].imshow(sample_R, cmap="Blues", vmin=0.0, vmax=1.0)
axes[0].set_title(f"Retention Matrix R (Naive Sequential, Seed {results_naive[0]['seed']})", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Evaluated Task Index", fontsize=10)
axes[0].set_ylabel("Trained Task Milestone", fontsize=10)
axes[0].set_xticks(range(config.num_tasks))
axes[0].set_yticks(range(config.num_tasks))
axes[0].set_xticklabels([f"Task {i+1}" for i in range(config.num_tasks)])
axes[0].set_yticklabels([f"Task {i+1}" for i in range(config.num_tasks)])

for i in range(config.num_tasks):
    for j in range(config.num_tasks):
        axes[0].text(j, i, f"{sample_R[i, j]:.2f}", ha="center", va="center",
                     color="white" if sample_R[i, j] > 0.55 else "black", fontsize=9)
cbar = plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
cbar.set_label("Accuracy", fontsize=9)

# Plot 2: Retention Decay of Task 1 vs Subsequent Task Training
task1_retention = [sample_R[step, 0] for step in range(config.num_tasks)]
task_steps = list(range(1, config.num_tasks + 1))
axes[1].plot(task_steps, task1_retention, marker='o', linewidth=2.5, color='#c0392b', label="Naive: Task 1 Retention")
axes[1].axhline(y=summary_joint['A_T_mean'], color='#2c3e50', linestyle='--', linewidth=2, 
                label=f"Joint Oracle Mean ({summary_joint['A_T_mean']:.2f})")
axes[1].axhline(y=0.10, color='gray', linestyle=':', label="Random Chance (10%)")
axes[1].set_title("Catastrophic Forgetting: Task 1 Performance Degradation", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Number of Sequentially Learned Tasks", fontsize=10)
axes[1].set_ylabel("Accuracy on Task 1", fontsize=10)
axes[1].set_ylim(-0.02, 1.05)
axes[1].set_xticks(task_steps)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc="upper right", frameon=True)

plt.tight_layout()

# Save figure to repository directory structure
fig_dir = os.path.join(config.output_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "EXP-2026-001_retention_curves.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"High-resolution figure saved to: {fig_path}")
plt.show()

In [ ]:
# ==============================================================================
# Result Serialization to JSON & CSV
# ==============================================================================
metrics_dir = os.path.join(config.output_dir, "metrics")
os.makedirs(metrics_dir, exist_ok=True)

artifact_payload = {
    "experiment_id": config.experiment_id,
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": {
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if cuda_available else "CPU",
        "gpu_memory_gb": total_memory_gb,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda if cuda_available else None
    },
    "configuration": asdict(config),
    "results": {
        "naive_sequential": summary_naive,
        "joint_offline_oracle": summary_joint
    }
}

json_path = os.path.join(metrics_dir, "EXP-2026-001_metrics.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(artifact_payload, f, indent=2)

print(f"Metrics JSON exported: {json_path}")

## 6. Scientific Analysis & Hypothesis Verification

Upon execution of the experiment above, evaluate findings against the formal preregistered criteria:

1. **Test of Primary Hypothesis ($H_1$):**
   - Did $BWT_{\text{naive}} \le -0.70$?  
   - Did $A_{5, \text{joint}} \ge 0.90$?  
   - If both hold, $H_1$ is **CONFIRMED**, quantitatively establishing catastrophic forgetting under sequential fine-tuning as an empirical lower bound for Track A continual learning research.
   - If $BWT_{\text{naive}} > -0.15$, $H_0$ cannot be rejected, and the result must be flagged as **DISCONFIRMED** in `experiments/register.md`.

2. **Repository Updating Checklist:**
   - [ ] Record final metrics ($A_T, BWT, FWT$) in [`experiments/register.md`](../register.md).
   - [ ] Update status in [`experiments/protocols/EXP-2026-001-protocol.md`](../protocols/EXP-2026-001-protocol.md) from `PLANNED` to `COMPLETED` (or `DISCONFIRMED`).
   - [ ] Commit generated artifacts (`metrics/*.json`, `figures/*.png`) using Git:
     ```bash
     git add experiments/
     git commit -m "exp(EXP-2026-001): record empirical continual learning baseline metrics and figures"
     ```